# Transformer(10. 마지막 단계 Linear Layer -> Softmax) 언어 모델 총정리 

## 1. Decoder Block
	◦ 트랜스포머 디코더 블록 단계별 공식
	1) 입력값(token) 
	2) 임베딩 + 위치인코딩 
	3) Masked Multi-Head Attention 
	4) 1차 Residual Connection → 1차 Layer Normalization
	5) Cross Attention(인코더 Key/Value, 디코더 Query 사용하여 Attention 계산) 
	6) 2차 Residual Connection → 2차 Layer Normalization 
	7) Feed Forward Network(FFN)
	8) 3차 Residual Connection → 3차 Layer Normalization
	9) Linear Layer(선형계층)
	10) Softmax(소프트맥스 확률분포)

---
### 1) 임베딩 + 위치 인코딩
$ [ Y = \text{Embedding}(tokens) + \text{PositionalEncoding} ] $

### 2) Masked Multi-Head Attention
$ [ \text{MaskedMHA}(Y) = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + \text{mask}\right)V ] $

### 3) Residual Connection + Layer Normalization (1차)
$ [ H_1 = \text{LayerNorm}(Y + \text{MaskedMHA}(Y)) ] $

### 4) Cross Attention (인코더 Key/Value, 디코더 Query 사용하여 Attention 계산)
$ [ \text{CrossAttn}(H_1, H_{enc}) = \text{Softmax}\left(\frac{Q_{dec}K_{enc}^\top}{\sqrt{d_k}}\right)V_{enc} ] $

### 5) Residual Connection + Normalization (2차)
$ [ H_2 = \text{LayerNorm}(H_1 + \text{CrossAttn}(H_1, H_{enc})) ] $

### 6) Feed Forward Network (FFN)
$ [ \text{FFN}(H_2) = \max(0, H_2 W_1 + b_1) W_2 + b_2 ] $

### 7) Residual Connection + LayerNorm (3차)
$ [ H_3 = \text{LayerNorm}(H_2 + \text{FFN}(H_2)) ] $

### 8) Linear Layer(선형계층)
$ [ z = x W_o + b_o ] $

### 9) Softmax(소프트맥스 확률분포)
$ [ P_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}} ] $

---

## 2. Transformer(10. 마지막 단계 Linear Layer -> Softmax) 언어 모델의 10. 마지막 단계 Linear Layer -> Softmax 계산 과정을 공식과 함께 단계별로 계산 
---
### 디코더 최종 출력 단계
### 1) 입력값
$ • 디코더 블록 최종 출력 (3차 Residual + LayerNorm 결과):
[ x = [-1.0, 1.0] ] $

---
### 2) 선형 계층 (Linear Layer)
$ • 목적: 디코더 출력 벡터를 **어휘 집합 크기 [ V ]**에 맞게 변환 $

$ • 가중치 행렬:
[ W_o \in \mathbb{R}^{d \times V}, \quad b_o \in \mathbb{R}^V ] $

$ 예시로 [ d=2 ], [ V=3 ] (어휘 크기 3개 단어)라고 가정합니다. $

$ [ W_o =
\begin{bmatrix}
0.2 & -0.1 & 0.4 \\
-0.3 & 0.5 & 0.1
\end{bmatrix}, \quad
b_o = [0.0, 0.0, 0.0] ] $ 

계산:
$ [ z = x W_o + b_o
 ]
[ = [-1.0, 1.0] \cdot
\begin{bmatrix}
0.2 & -0.1 & 0.4 \\
-0.3 & 0.5 & 0.1
\end{bmatrix} ] $

$ • 첫 번째 성분: [ (-1)(0.2) + (1)(-0.3) = -0.2 - 0.3 = -0.5 ] $

$ • 두 번째 성분: [ (-1)(-0.1) + (1)(0.5) = 0.1 + 0.5 = 0.6 ] $

$ • 세 번째 성분: [ (-1)(0.4) + (1)(0.1) = -0.4 + 0.1 = -0.3 ] $

따라서:
$ [ z = [-0.5, 0.6, -0.3] ] $

---
### 3) 소프트맥스 (Softmax)
$ • 공식:
[ P_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}} ] $

계산:

$ • [ e^{-0.5} \approx 0.6065 ] $

$ • [ e^{0.6} \approx 1.8221 ] $

$ • [ e^{-0.3} \approx 0.7408 ] $

합:

$ [ 0.6065 + 1.8221 + 0.7408 = 3.1694 ] $

확률:

$ • 첫 번째 : [ 0.6065 / 3.1694 \approx 0.191 ] $

$ • 두 번째 : [ 1.8221 / 3.1694 \approx 0.575 ] $

$ • 세 번째 : [ 0.7408 / 3.1694 \approx 0.234 ] $

---
✅ 최종 결론

$ • 선형 계층 출력 (logits): [ [-0.5, 0.6, -0.3] ] $

$ • Softmax 확률 분포: [ [0.191, 0.575, 0.234] ] $

$ • 즉, 모델은 두 번째 단어를 가장 높은 확률(57.5) $ % 로 예측합니다.

---